# 24.2 Genetic programming

Genetic programming evolves executable expressions, not just numeric vectors.

A candidate is a program tree whose predictions are scored on data. Subtree crossover and mutation preserve arity, while parsimony pressure prices tree size so symbolic search does not win by bloat.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 242
rng = np.random.default_rng(SEED)
OPS = ("add", "sub", "mul", "sin", "cos")


def variable(index=0):
    return ("var", index)


def constant(value):
    return ("const", float(value))


def node(op, left=None, right=None):
    if op in {"sin", "cos"}:
        return (op, left)
    return (op, left, right)


def tree_size(tree):
    tag = tree[0]
    if tag in {"var", "const"}:
        return 1
    if tag in {"sin", "cos"}:
        return 1 + tree_size(tree[1])
    return 1 + tree_size(tree[1]) + tree_size(tree[2])


def eval_tree(tree, x):
    tag = tree[0]
    if tag == "var":
        return x[:, tree[1]]
    if tag == "const":
        return np.full(x.shape[0], tree[1])
    if tag == "add":
        return eval_tree(tree[1], x) + eval_tree(tree[2], x)
    if tag == "sub":
        return eval_tree(tree[1], x) - eval_tree(tree[2], x)
    if tag == "mul":
        return eval_tree(tree[1], x) * eval_tree(tree[2], x)
    if tag == "sin":
        return np.sin(eval_tree(tree[1], x))
    if tag == "cos":
        return np.cos(eval_tree(tree[1], x))
    raise ValueError(tag)


def mse(tree, x, y):
    prediction = eval_tree(tree, x)
    return float(np.mean(np.square(prediction - y)))


def score_tree(tree, x, y, parsimony=0.05):
    return -(mse(tree, x, y) + parsimony * tree_size(tree))


def random_tree(rng, dim, depth):
    if depth <= 0 or rng.random() < 0.25:
        if rng.random() < 0.7:
            return variable(int(rng.integers(0, dim)))
        return constant(float(rng.choice([-2.0, -1.0, 0.0, 1.0, 2.0])))
    op = rng.choice(OPS)
    if op in {"sin", "cos"}:
        return node(op, random_tree(rng, dim, depth - 1))
    return node(op, random_tree(rng, dim, depth - 1), random_tree(rng, dim, depth - 1))


def tree_paths(tree, prefix=()):
    paths = [prefix]
    tag = tree[0]
    if tag in {"var", "const"}:
        return paths
    if tag in {"sin", "cos"}:
        paths.extend(tree_paths(tree[1], prefix + (1,)))
        return paths
    paths.extend(tree_paths(tree[1], prefix + (1,)))
    paths.extend(tree_paths(tree[2], prefix + (2,)))
    return paths


def get_subtree(tree, path):
    current = tree
    for step in path:
        current = current[step]
    return current


def replace_subtree(tree, path, replacement):
    if len(path) == 0:
        return replacement
    tag = tree[0]
    step = path[0]
    if tag in {"sin", "cos"}:
        return (tag, replace_subtree(tree[1], path[1:], replacement))
    if step == 1:
        return (tag, replace_subtree(tree[1], path[1:], replacement), tree[2])
    return (tag, tree[1], replace_subtree(tree[2], path[1:], replacement))


def subtree_crossover(parent_a, parent_b, rng):
    path_a = tree_paths(parent_a)[int(rng.integers(0, len(tree_paths(parent_a))))]
    path_b = tree_paths(parent_b)[int(rng.integers(0, len(tree_paths(parent_b))))]
    donor = get_subtree(parent_b, path_b)
    child = replace_subtree(parent_a, path_a, donor)
    return child


def mutate_tree(tree, rng, dim, max_depth=3):
    path = tree_paths(tree)[int(rng.integers(0, len(tree_paths(tree))))]
    replacement = random_tree(rng, dim, int(rng.integers(0, max_depth + 1)))
    return replace_subtree(tree, path, replacement)


def make_gp_ladder():
    x1 = np.array([[-1.0], [0.0], [1.0], [2.0]])
    y1 = np.square(x1[:, 0]) + 1.0
    grid = np.linspace(-2.0, 2.0, 7)
    x2 = np.array([[a, b] for a in grid for b in grid])
    y2 = np.square(x2[:, 0]) + np.square(x2[:, 1])
    x3 = x2.copy()
    y3 = np.sin(3.0 * x3[:, 0]) + np.cos(2.0 * x3[:, 1]) + 0.1 * np.square(x3[:, 0])
    x4 = x2.copy()
    y4 = np.square(x4[:, 0] - 1.0) + np.square(x4[:, 1] + 1.0)
    x5_rng = np.random.default_rng(SEED + 5)
    x5 = x5_rng.uniform(-1.0, 1.0, size=(120, 8))
    y5 = x5[:, 0] * x5[:, 1] + np.sin(2.0 * x5[:, 2]) - np.square(x5[:, 3]) + 0.5 * x5[:, 4]
    return [
        {"name": "D1 known x^2+1", "x": x1, "y": y1, "dim": 1, "parsimony": 0.05},
        {"name": "D2 2-D sphere expression", "x": x2, "y": y2, "dim": 2, "parsimony": 0.04},
        {"name": "D3 multimodal symbolic target", "x": x3, "y": y3, "dim": 2, "parsimony": 0.04},
        {"name": "D4 size-constrained objective", "x": x4, "y": y4, "dim": 2, "parsimony": 0.12},
        {"name": "D5 high-dimensional feature expression", "x": x5, "y": y5, "dim": 8, "parsimony": 0.06},
    ]


def gp_optimize(x, y, dim, population_size=60, generations=40, parsimony=0.05, max_depth=4, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)
    population = [random_tree(rng, dim, max_depth) for _ in range(population_size)]
    history = []
    for generation in range(generations):
        scores = np.array([score_tree(tree, x, y, parsimony) for tree in population])
        order = np.argsort(scores)[::-1]
        elite_count = max(2, population_size // 10)
        next_population = [population[index] for index in order[:elite_count]]
        probabilities = scores - np.min(scores) + 1.0
        probabilities = probabilities / np.sum(probabilities)
        while len(next_population) < population_size:
            parents = rng.choice(population_size, size=2, replace=True, p=probabilities)
            child = subtree_crossover(population[int(parents[0])], population[int(parents[1])], rng)
            if rng.random() < 0.35:
                child = mutate_tree(child, rng, dim, max_depth=max_depth)
            next_population.append(child)
        population = next_population
        history.append(float(scores[order[0]]))
    scores = np.array([score_tree(tree, x, y, parsimony) for tree in population])
    best_index = int(np.argmax(scores))
    return {"best": population[best_index], "best_fitness": float(scores[best_index]), "history": np.array(history), "population": population}


## The concept, built once (D1): score behavior plus parsimony

The lesson fitness is

$$F(T)=-\frac{1}{m}\sum_i (T(x_i)-y_i)^2-\lambda |T|.$$

For $x=\{-1,0,1,2\}$ and $y=x^2+1$, the lesson MSEs are $2.0$, $0.0$, and $2.5$.

In [ ]:

x_demo = np.array([[-1.0], [0.0], [1.0], [2.0]])
y_demo = np.square(x_demo[:, 0]) + 1.0

t1 = node("add", variable(0), constant(1.0))
t2 = node("add", node("mul", variable(0), variable(0)), constant(1.0))
t3 = node("add", node("mul", constant(2.0), variable(0)), constant(1.0))

mse_values = np.array([mse(t1, x_demo, y_demo), mse(t2, x_demo, y_demo), mse(t3, x_demo, y_demo)])
print("MSEs:", mse_values)

assert np.allclose(mse_values, np.array([2.0, 0.0, 2.5]))


## Closure-safe operators and the lesson's parsimony scores

With $\lambda=0.05$, an exact size-5 tree scores $0.25$, an exact size-9 tree scores $0.45$, and a size-3 tree with raw error $0.25$ scores $0.40$. The optimizer uses those same scores while keeping every subtree arity-valid.

In [ ]:

parsimony = 0.05
lesson_scores = np.array([0.0 + parsimony * 5.0, 0.0 + parsimony * 9.0, 0.25 + parsimony * 3.0])

batch_invalid = 0
for _ in range(50):
    child = subtree_crossover(t2, t3, rng)
    try:
        eval_tree(child, x_demo)
    except Exception:
        batch_invalid += 1

print("lesson parsimony scores:", lesson_scores)
print("invalid crossover children:", batch_invalid)

assert np.allclose(lesson_scores, np.array([0.25, 0.45, 0.40]))
assert batch_invalid == 0


## The dataset ladder

The inline F4 ladder moves from known symbolic regression to 2-D sphere-like expressions, noisy multimodal behavior, size-constrained scoring, and an 8-feature D5 expression search.

In [ ]:

ladder = make_gp_ladder()

for rung in ladder:
    print(rung["name"], "x shape", rung["x"].shape, "target sample", np.round(rung["y"][:5], 3), "lambda", rung["parsimony"])


In [ ]:

results = []

for index, rung in enumerate(ladder):
    result = gp_optimize(rung["x"], rung["y"], rung["dim"], population_size=60, generations=40, parsimony=rung["parsimony"], max_depth=4, rng=np.random.default_rng(SEED + index))
    results.append(result)
    print(f"{rung['name']}: best_fitness={result['best_fitness']:.4f}, size={tree_size(result['best'])}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 6))

for index, rung in enumerate(ladder):
    prediction = eval_tree(results[index]["best"], rung["x"])
    error = prediction - rung["y"]
    ax = axes[0, index]
    if rung["dim"] == 1:
        ax.plot(rung["x"][:, 0], rung["y"], label="target")
        ax.scatter(rung["x"][:, 0], prediction, label="program")
    else:
        ax.scatter(rung["x"][:, 0], rung["x"][:, 1], c=error, cmap="coolwarm", s=20)
    ax.set_title(rung["name"].split()[0])

for index, result in enumerate(results):
    axes[1, index].plot(result["history"])
    axes[1, index].set_title("best fitness")
    axes[1, index].set_xlabel("generation")

fig.suptitle("Expression error panels and best-fitness curves")
plt.tight_layout()


## Pitfall on D5: bloat without behavioral gain

Setting parsimony to zero lets neutral branches grow. The fix restores a validation-style score with positive parsimony pressure.

In [ ]:

d5 = ladder[-1]

bloated = gp_optimize(d5["x"], d5["y"], d5["dim"], population_size=60, generations=40, parsimony=0.0, max_depth=5, rng=np.random.default_rng(555))
regularized = gp_optimize(d5["x"], d5["y"], d5["dim"], population_size=60, generations=40, parsimony=d5["parsimony"], max_depth=4, rng=np.random.default_rng(555))

print("no parsimony size", tree_size(bloated["best"]), "fitness", bloated["best_fitness"])
print("fixed size", tree_size(regularized["best"]), "fitness", regularized["best_fitness"])


## Evaluate it + Practice

- Compare the reported best fitness against a seeded random-search baseline with the same evaluation budget.
- Sanity check: D1 must hit the lesson's worked calculation before trusting D2–D5.
- Ablation: set parsimony to zero and allow deeper trees should make the hardest rung worse or less stable.
- Failure signals: population variance collapses too early, bounds are violated, or the summary curve improves only by one lucky sample.
- Re-run with another seed only after the structural checks pass; do not tune from a single trace.

Practice prompts:
1. Change the hardest rung budget and explain whether convergence improves per objective call.
2. Add one diagnostic for diversity and plot it next to the metric.
3. Swap the D3 multimodal objective and predict which operator needs retuning.